# Video Editing flower run: red bud → blue bud with reconstruction
One edit with the colour defaults from the demo sweep; saves source, reconstruction and edited frames.

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, time, zipfile, urllib.request
from pathlib import Path
candidates = [p.parent for p in Path('/kaggle/input').rglob('launch.py') if (p.parent / 'run_extended.py').is_file()]
assert len(candidates) == 1, f'Attach fatezero-video-editing-code-v2 (found {len(candidates)})'
PROJECT = Path('/tmp/fz_code'); shutil.copytree(candidates[0], PROJECT, dirs_exist_ok=True)
WORK = Path('/kaggle/working'); RUNS = WORK / 'runs'; RUNS.mkdir(exist_ok=True, parents=True)
BASE = Path('/tmp/video_editing_runtime'); BASE.mkdir(exist_ok=True)
ENV = os.environ.copy()
ENV.update(PYTHONNOUSERSITE='1', TOKENIZERS_PARALLELISM='false', HF_HOME=str(BASE / 'huggingface'),
           HF_HUB_DISABLE_TELEMETRY='1', UV_PYTHON_INSTALL_DIR=str(BASE / 'managed_python'),
           UV_CACHE_DIR=str(BASE / 'uv_cache'), CUDA_VISIBLE_DEVICES='0', PYTHONUNBUFFERED='1', SAM2_BUILD_CUDA='0')
ENV.pop('PYTHONPATH', None)
uv = BASE / 'bootstrap' / 'bin' / 'uv'
if not uv.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--target', str(BASE / 'bootstrap'), 'uv==0.6.17'], check=True)
venv = BASE / 'modern_venv'; python = venv / 'bin' / 'python'
if not python.exists():
    subprocess.run([str(uv), 'venv', '--python', '3.10.16', str(venv)], check=True, env=ENV)
    subprocess.run([str(uv), 'pip', 'install', '--python', str(python), '-r', str(PROJECT / 'requirements-modern.txt'),
                    '--index-url', 'https://pypi.org/simple', '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
                    '--index-strategy', 'unsafe-best-match'], check=True, env=ENV)
    subprocess.run([str(uv), 'pip', 'install', '--python', str(python), '--no-deps', '-r',
                    str(PROJECT / 'requirements-segmentation.txt')], check=True, env=ENV)
subprocess.run([str(python), '-c', 'import torch,diffusers;print(torch.__version__,diffusers.__version__,torch.cuda.get_device_name(0))'],
               check=True, env=ENV)

In [ ]:
CLIPS = {'flower': ('url', 'https://interactive-examples.mdn.mozilla.net/media/cc0-videos/flower.mp4', None, 0)}
JOB = {'name': 'flower_blue_reconstruction', 'clip': 'flower', 'source': 'a red flower bud on a green stem with purple-edged leaves, blurred garden background', 'target': 'a blue flower bud on a green stem with purple-edged leaves, blurred garden background', 'edit_words': ['flower', 'bud'], 'mask_backend': 'attention', 'settings': {'self_replace': 0.7, 'cross_replace': 0.5, 'guidance': 7.5, 'negative_prompt_inversion': True, 'mask_latent_blend': True, 'blend_dilate': 1, 'reconstruct': True, 'stream_profile': False}}
DATA = Path('/tmp/fz_data'); DATA.mkdir(exist_ok=True)
for z in ('attribute', 'style'):
    if not (DATA / z).exists():
        urllib.request.urlretrieve(f'https://github.com/ChenyangQiQi/FateZero/releases/download/v0.0.1/{z}.zip', DATA / f'{z}.zip')
        zipfile.ZipFile(DATA / f'{z}.zip').extractall(DATA / z)
VIDEOS = WORK / 'clips'; VIDEOS.mkdir(exist_ok=True)
def clip_path(name):
    kind, where, stride, start = CLIPS[name]
    target = VIDEOS / f'{name}.mp4'
    if target.exists():
        return target
    if kind == 'url':
        urllib.request.urlretrieve(where, target)
        return target
    folder = next(p for p in (DATA / kind).rglob(where) if p.is_dir())
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in ('.png', '.jpg', '.jpeg'))[start::stride][:8]
    subprocess.run([str(python), '-c', f"""
import imageio.v2 as imageio, numpy as np
from PIL import Image
files = {[str(f) for f in files]!r}
with imageio.get_writer({str(target)!r}, fps=8, codec='libx264', quality=10, pixelformat='yuv420p', macro_block_size=None) as w:
    for f in files:
        im = Image.open(f).convert('RGB'); s = min(im.size)
        im = im.crop(((im.width - s) // 2, (im.height - s) // 2, (im.width + s) // 2, (im.height + s) // 2)).resize((512, 512), Image.LANCZOS)
        w.append_data(np.asarray(im))
"""], check=True, env=ENV)
    return target

In [ ]:
import json, time, subprocess
from pathlib import Path
base = json.loads((PROJECT / 'configs' / 'improved_sd14.json').read_text())
j = JOB
out = WORK / 'flower' / j['name']
c = dict(base, video_url='', video_path=str(clip_path('flower')), clip_seconds=1.0, start_frame=0, frame_stride=1,
         frames=8, size=512, steps=30, inversion_iterations=3, seed=0, source_prompt=j['source'], target_prompt=j['target'],
         edit_words=j['edit_words'], mask_backend=j['mask_backend'], mask_prompt=' '.join(j['edit_words']),
         cache_path='/tmp/fz_cache', mask_cache_path='/tmp/fz_mask_cache', keep_cache=False, reuse_cache=False,
         output_dir=str(out), edit_profile='improved', negative_prompt='')
c.update(j['settings'])
out.mkdir(parents=True, exist_ok=True)
(out / 'request.json').write_text(json.dumps(c, indent=2))
start = time.time()
with (out / 'log.txt').open('w') as log:
    code = subprocess.run([str(python), str(PROJECT / 'launch.py'), '--config', str(out / 'request.json')],
                          cwd=PROJECT, env=ENV, stdout=log, stderr=subprocess.STDOUT).returncode
print('exit', code, f'{time.time() - start:.0f}s', flush=True)
if code:
    print((out / 'log.txt').read_text()[-3000:], flush=True)
    raise SystemExit(code)

In [ ]:
import cv2, numpy as np
from PIL import Image, ImageDraw
out = WORK / 'flower' / JOB['name']
subprocess.run([str(python), str(PROJECT / 'metrics.py'), str(out), '--output', str(out / 'metrics.json')],
               cwd=PROJECT, env=ENV, check=True)
def frames_from_video(path):
    capture = cv2.VideoCapture(str(path)); frames = []
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    capture.release()
    return frames
rows = [('source_frames', sorted((out / 'source_frames').glob('*.png'))),
        ('reconstruction_frames', frames_from_video(out / 'reconstruction.mp4')),
        ('edited_frames', sorted((out / 'edited_frames').glob('*.png')))]
rows = [(name, [Image.open(p).convert('RGB') if isinstance(p, Path) else p for p in items]) for name, items in rows]
T, LABEL = 320, 24
pick = [0, 2, 4, 6]
grid = Image.new('RGB', (T * len(pick), (T + LABEL) * len(rows)), 'white')
draw = ImageDraw.Draw(grid)
for r, (name, items) in enumerate(rows):
    draw.text((4, r * (T + LABEL) + 6), name, fill='black')
    for c, i in enumerate(pick):
        grid.paste(items[i].resize((T, T)), (c * T, r * (T + LABEL) + LABEL))
grid.save(WORK / 'flower_blue_comparison.png')
print((out / 'metrics.json').read_text()[:1500], flush=True)
print('GRID_DONE', flush=True)